In [ ]:
####################### Old Code #############################################################################
import os
import time
import os.path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager  # Optional, for automatic driver management
service = Service(ChromeDriverManager().install())  # or specify your path directly here

# Pass the Service object to the WebDriver
driver = webdriver.Chrome(service=service)
# driver.get("https://www.tofler.in/")

# input_list = ["Ram Ganga Valley Chemicals & Fertilisers Limited", 
#               "Susmit Sangita Intermagnetic Private Limited",
#               "Teletek Electronics (Private) Limited", 
#               "Vaishali Mineral Water, Private Limited",
#               "Verma Departmental Stores Private Limited",
#                 "S K G Consolidated Limited",
#                   "Dutch Ply Industries Limited",
#               "S D Jewellers Private Limited",
#                 "Rajsthan Ambuja Industries Private Limited"]
input_list = pd.read_excel(r"D:\\Nexensus_Projects\\pdfFoms\\Book2.xlsx")['c_name'].to_list()[664:]
all_data = []
for input_name in input_list:
    try:
        driver.get("https://www.tofler.in/")
        time.sleep(2)
        driver.find_element(By.ID, "searchbox").send_keys(input_name.strip().rstrip(','))
        time.sleep(4)
        driver.find_element(By.ID, "ui-id-1").click()
        time.sleep(2)
        cin = driver.current_url.split("/")[-1]
        corrected_name = driver.find_element(By.CLASS_NAME, "company_title.items-center.nowrap.gap-16").text.strip()
        # print("cin :", cin)
        # print("correct_name :", corrected_name)
        print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
        all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})
        print("\n")
    except:
        print("failed now trying with cleared name")
        try:
            driver.get("https://www.tofler.in/")
            time.sleep(2)
            driver.find_element(By.ID, "searchbox").send_keys(input_name.lower().replace("limited.", "").replace("private.","").replace("limited", "").replace("private","").replace("pvt.", "").replace("ltd.", "").replace("pvt", "").replace("ltd", "").replace("llp.", "").replace("llp", "").replace("messers", "").replace("messars", "").replace("m/s", "").replace("()", "").strip().rstrip(','))
            time.sleep(4)
            driver.find_element(By.ID, "ui-id-1").click()
            time.sleep(2)
            cin = driver.current_url.split("/")[-1]
            corrected_name = driver.find_element(By.CLASS_NAME, "company_title.items-center.nowrap.gap-16").text.strip()
            # print("cin :", cin)
            # print("correct_name :", corrected_name)
            print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
            all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})
            print("\n")
        except Exception as e:            
            print("Not found :", input_name)
            all_data.append({"input": input_name, "corrected_name": None, "cin": None, })

driver.quit()
if all_data:
    data_df = pd.DataFrame(all_data).reset_index(drop=True)

    # Optional: Save output for each file
    output_path = os.path.join(f"Tofler_data_6.xlsx")
    data_df.to_excel(output_path, index=False)
    print("💾 Saved:", output_path)

failed now trying with cleared name
{'input': 'VVMN PRIVATE LIMITED', 'cin': 'U47219KA2023PTC172213', 'corrected_name': 'VVMN PRIVATE LIMITED'}


{'input': 'MAHANADU NELAMANE AGRO PRIVATE LIMITED', 'cin': 'U46101KA2023PTC172284', 'corrected_name': 'MAHANADU NELAMANE AGRO PRIVATE LIMITED'}


{'input': 'SYD INNOVATION TECHNOLOGY SOLUTIONS (OPC) PRIVATE LIMITED', 'cin': 'U62013KA2023OPC171384', 'corrected_name': 'SYD INNOVATION TECHNOLOGY SOLUTIONS (OPC) PRIVATE LIMITED'}


{'input': 'S G FRUIT TRADING PRIVATE LIMITED', 'cin': 'U46301KA2023PTC172651', 'corrected_name': 'S G FRUIT TRADING PRIVATE LIMITED'}


{'input': 'MAKAM ELITE STEELS PRIVATE LIMITED', 'cin': 'U24108KA2023PTC172653', 'corrected_name': 'MAKAM ELITE STEELS PRIVATE LIMITED'}


failed now trying with cleared name
Not found : ARZTIC CONSULTING (OPC) PRIVATE LIMITED
{'input': 'HENGFEI INDIA GARMENTSPRIVATE LIMITED', 'cin': 'U14101KA2023PTC173615', 'corrected_name': 'HENGFEI INDIA GARMENTSPRIVATE LIMITED'}


{'input': 'AMISU C

In [ ]:
######################### Batch wise processing for tofler ####################################################
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import re


# === List of realistic user agents (desktop + mobile mix) ===
USER_AGENTS = [
    # Chrome (Windows)
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    # # Chrome (Mac)
    # "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    # "(KHTML, like Gecko) Chrome/118.0.5993.90 Safari/537.36",
    # Edge
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/118.0.5993.90 Safari/537.36 Edg/118.0.2088.61",
    # Firefox
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0",
    # # Mobile (Android Chrome)
    # "Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 "
    # "(KHTML, like Gecko) Chrome/119.0.6045.134 Mobile Safari/537.36",
    # # Mobile (iPhone)
    # "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) AppleWebKit/605.1.15 "
    # "(KHTML, like Gecko) Version/17.0 Mobile/15E148 Safari/604.1",
]

def human_delay(a=2, b=5):
    """Random sleep between a and b seconds."""
    time.sleep(random.uniform(a, b))

def start_driver():
    """Initialize and return a new Chrome WebDriver instance with random user-agent."""
    ua = random.choice(USER_AGENTS)
    print(f"🧠 Using User-Agent: {ua}\n")

    service = Service(ChromeDriverManager().install())
    options = Options()

    # --- Headless Mode ---
    options.add_argument("--headless=new")     # Recommended for Chrome 109+
    options.add_argument("--window-size=1920,1080")

    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument(f"user-agent={ua}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(service=service, options=options)
    return driver

def extract_after_manager(simplified: str):
    # Split on "manager" (case-insensitive), only once
    parts = re.split(r'manager', simplified, flags=re.IGNORECASE)
    
    if len(parts) < 2:
        return simplified  # or return None if you prefer
    
    result = parts[1]

    # Remove leading spaces, commas, periods
    result = result.lstrip(" ,.")

    return result.strip()
# === Load Input Data ===
import pandas as pd
# df_1 = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\NCLT_17-11-2025_350-975.xlsx")
# df_2 = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\NCLT_17-11-2025_975-1175.xlsx")
# df_3 = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\NCLT_17-11-2025_1175-1325.xlsx")
# df_4 = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\NCLT_17-11-2025_data_0-350.xlsx")
# # df_5 = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\Tofler_data_1.xlsx")
# df = pd.concat([df_1, df_2, df_3, df_4])
# df = df.drop_duplicates()
# input_list = df[df['cin'].isna()]['input'].to_list()
input_list = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\input_folder\WD For Kripal singh.xlsx")['c_name'].drop_duplicates().to_list()[:]

# === Output File ===
output_path = "WD_data.xlsx"
all_data = []

# === Resume if partial file exists ===
if os.path.exists(output_path):
    existing_df = pd.read_excel(output_path)
    processed_names = set(existing_df['input'])
    input_list = [name for name in input_list if name not in processed_names]
    all_data = existing_df.to_dict('records')
    print(f"🟡 Resuming — already processed {len(processed_names)} companies.")
else:
    print("🟢 Starting fresh extraction.")

batch_size = 50
driver = start_driver()

for idx, input_name in enumerate(input_list, start=1):
    try:
        driver.get("https://www.tofler.in/")
        time.sleep(2)

        # Clean company name for first attempt
        clean_name = input_name.strip().rstrip(',')
        driver.find_element(By.ID, "searchbox").send_keys(clean_name)
        time.sleep(4)
        driver.find_element(By.ID, "ui-id-1").click()
        time.sleep(2)

        cin = driver.current_url.split("/")[-1]
        corrected_name = driver.find_element(
            By.CLASS_NAME, "company_title.items-center.nowrap.gap-16"
        ).text.strip()

        # print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
        all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})
        # print()

    except Exception:
        print("❌ Failed — retrying with simplified name:", input_name)
        try:
            driver.get("https://www.tofler.in/")
            time.sleep(2)

            simplified = (
                input_name.lower()
                .replace("limited.", "")
                .replace("private.", "")
                .replace("limited", "")
                .replace("private", "")
                .replace("pvt.", "")
                .replace("ltd.", "")
                .replace("pvt", "")
                .replace("ltd", "")
                .replace("llp.", "")
                .replace("llp", "")
                .replace("messers", "")
                .replace("messars", "")
                .replace("m/s", "")
                .replace("()", "")
                # .replace("A.O.,", "")
                .strip()
                .rstrip(',')
            )
            simplified = extract_after_manager(simplified)
            if "officer," in simplified:
                simplified = simplified.split("officer,")[1].strip()
            elif "officer ," in simplified:
                simplified = simplified.split("officer,")[1].strip()
            driver.find_element(By.ID, "searchbox").send_keys(simplified)
            time.sleep(4)
            driver.find_element(By.ID, "ui-id-1").click()
            time.sleep(2)

            cin = driver.current_url.split("/")[-1]
            corrected_name = driver.find_element(
                By.CLASS_NAME, "company_title.items-center.nowrap.gap-16"
            ).text.strip()

            # print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
            all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})
            # print()

        except Exception as e:
            # print("🚫 Not found:", input_name)
            all_data.append({"input": input_name, "corrected_name": None, "cin": None})

    # === Restart driver + Save progress after every 50 ===
    if idx % batch_size == 0:
        print(f"\n💾 Saving progress and restarting browser after {idx} companies...\n")
        pd.DataFrame(all_data).to_excel(output_path, index=False)
        print(f"✅ Progress saved to {output_path}")

        driver.quit()
        time.sleep(20)
        driver = start_driver()
    if idx % 200 == 0:
        driver.quit()
        print("sleeping for 5 minutes for recovery")
        # time.sleep(300)
        time.sleep(random.uniform(100, 200))
        driver = start_driver()

# === Final save ===
driver.quit()
if all_data:
    pd.DataFrame(all_data).to_excel(output_path, index=False)
    print(f"🎉 All done! Final file saved to {output_path}")


🟢 Starting fresh extraction.
🧠 Using User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0

❌ Failed — retrying with simplified name: "Vaidnath Sah Sakhar Karkhana Ltd
❌ Failed — retrying with simplified name: 1. Shri L. Madhusudhan Rao S/O Shri L. Venkata Ramanaidu And 
2. Smt L. Ramalakshmamma W/O Shri Lagadapati Rama Naidu 

(Guarantor Of Lanco Babandh Power Ltd
❌ Failed — retrying with simplified name: A&Z Lifestyle Pvt Ltd
❌ Failed — retrying with simplified name: Aarti Plantation & Agro Products Ltd
❌ Failed — retrying with simplified name: Ab Bank Ltd
❌ Failed — retrying with simplified name: Abhishek Transtel Ltd
❌ Failed — retrying with simplified name: Abhyudaya Co-Op. Bank Ltd
❌ Failed — retrying with simplified name: Act Farwarders Pvt Ltd
❌ Failed — retrying with simplified name: Adl International Ltd
❌ Failed — retrying with simplified name: Ads Hightech Polymers Ltd
❌ Failed — retrying with simplified name: Agrobel Industries Ltd
❌ 

In [22]:
input_name = "Authorised Officer Cum Chief Manager, Bank Of Baroda, Rasarb Branch"
simplified = (
    input_name.lower()
    .replace("limited.", "")
    .replace("private.", "")
    .replace("limited", "")
    .replace("private", "")
    .replace("pvt.", "")
    .replace("ltd.", "")
    .replace("pvt", "")
    .replace("ltd", "")
    .replace("llp.", "")
    .replace("llp", "")
    .replace("messers", "")
    .replace("messars", "")
    .replace("m/s", "")
    .replace("()", "")
    .replace("A.O.,", "")
    .strip()
    .rstrip(',')
)
import re

def extract_after_manager(simplified: str):
    # Split on "manager" (case-insensitive), only once
    parts = re.split(r'manager', simplified, flags=re.IGNORECASE)
    
    if len(parts) < 2:
        return simplified  # or return None if you prefer
    
    result = parts[1]

    # Remove leading spaces, commas, periods
    result = result.lstrip(" ,.")

    return result.strip()  # optional final cleanup

extract_after_manager(input_name)

'Bank Of Baroda, Rasarb Branch'

In [ ]:
import pandas as pd
df_2 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_2.xlsx")
df_3 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_3.xlsx")
df_4 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_4.xlsx")
df_5 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_5.xlsx")
df_6 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_6.xlsx")
df_7 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_7.xlsx")
df_8 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_8.xlsx")
df_9 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_9.xlsx")
df_10 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_10.xlsx")
df_11 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_11.xlsx")
df_12 = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\extra_tofler_data\Tofler_data_12.xlsx")
final_df = pd.concat([df_2, df_3, df_4, df_5, df_6, df_7, df_8, df_9, df_10, df_11, df_12])

# Create new column (case-insensitive comparison)
# df['is_match'] = df['input'].str.lower() == df['corrected_name'].str.lower()
final_df['is_match'] = final_df['input'].fillna('').str.lower() == final_df['corrected_name'].fillna('').str.lower()
final_df.to_excel("Tofler_data_13.xlsx", index=False)

In [ ]:
############# Code for adding threshold in single dataframe ##################################
from fuzzywuzzy import fuzz
import pandas as pd

ratio_list = []
partial_ratio_list = []
final_df = pd.read_excel(r"D:\\Nexensus_Projects\\ToflerAndZuba\\WD_Tofler_data.xlsx")
# final_df['corrected_name'] = final_df['corrected_name'].fillna('N/A')

for row in final_df.itertuples():
    if type(row.corrected_name) == float:
        ratio_list.append(0)  
    else:
        ratio_list.append(fuzz.ratio(row.input.lower(), row.corrected_name.lower())) 
    # ratio_list.append(fuzz.ratio(row.input.lower(), row.corrected_name.lower()))           # e.g., 88
    # partial_ratio_list.append(fuzz.partial_ratio(row.input.lower(), row.corrected_name.lower()))  
    # print("------------------------------------------------")

final_df['ratio'] = ratio_list
# final_df['partial_ratio'] = partial_ratio_list

final_df.to_csv("WDTest.csv")


In [5]:
import pandas as pd
final_df = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\Book6_data.xlsx")
final_df['is_match'] = final_df['input'].fillna('').str.lower() == final_df['corrected_name'].fillna('').str.lower()
final_df.to_excel("Book6_data.xlsx", index=False)

In [3]:
current_url = "https://www.zaubacorp.com/KSN-POLYCOATERS-LLP-AAJ-4175"
current_url.split("/")[-1].rsplit("-", 1)[0].rsplit("-", 1)[1]

'AAJ'

In [30]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd

df = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\output_folder\08-12-2025_data.xlsx")[:]

previous_names_list = []
old_name_match_list = []   # <-- store match result for each row

for row in df.itertuples(index=False):
    cin = row.cin
    input_name = row.input

    if isinstance(cin, float) or cin is None:
        previous_names_list.append([])
        old_name_match_list.append(False)   # no CIN → no match
        continue

    print(row.corrected_name)

    url = f"https://www.zaubacorp.com/companysearchresults/{cin}"
    company_page_response = requests.get(
        url,
        impersonate="chrome",
        cookies={}
    )

    soup = BeautifulSoup(company_page_response.text, 'html.parser')
    all_rc_divs = soup.find_all("div", class_='rc')

    previous_names = []
    for rc_div in all_rc_divs:
        h3 = rc_div.find('h3')
        if h3 and h3.text.strip() == "Previous Names":
            table = rc_div.find("table")
            if table:
                for tr in table.find_all("tr"):
                    td = tr.find('td')
                    if td:
                        previous_names.append(td.text.strip())

    previous_names_list.append(previous_names)

    # Normalize strings for safe matching
    normalized_input = input_name.lower().strip()
    normalized_prev = [name.lower().strip() for name in previous_names]

    match = normalized_input in normalized_prev
    old_name_match_list.append(match)

    if match:
        print("MATCH:", normalized_input)
        print("IN:", normalized_prev)

# Add new columns
df["previous_names"] = previous_names_list
df["old_name_match"] = old_name_match_list

df.to_excel("updated_with_previous_names.xlsx", index=False)

print("New column added successfully!")


LULU INDIA SHOPPING MALL PRIVATE LIMITED
GANTON PROJECTS PRIVATE LIMITED
MATCH: ganton projects private limited
IN: ['ganton aviation private limited', 'ganton projects private limited']
SAMSON AND SONS BUILDERS AND DEVELOPERS PRIVATE LIMITED
MATCH: samson and sons builders and developers private limited
IN: ['samson and sons builders and developers private li mited', 'samson and sons builders and developers private limited', 'samson and sons builders and developersprivate limited']
SAMSON AND SONS BUILDERS AND DEVELOPERS PRIVATE LIMITED (old name)
MATCH: samson and sons builders and developers private limited
IN: ['samson and sons builders and developers private li mited', 'samson and sons builders and developers private limited', 'samson and sons builders and developersprivate limited']
ABG SHIPYARD LTD
PAI BROTHERS ENGINEERS PRIVATE LIMITED
MATCH: pai brothers engineers private limited
IN: ['pai brothers engineers private limited']
SHIVAM SIGN-TECH PRIVATE LIMITED.
SHIVAM SIGN-TECH 